In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
import illustris_python as il
from matplotlib.colors import LogNorm
import matplotlib.animation as animation

In [ ]:
zs = [0.0,0.1,0.2,0.3,0.4,0.5,0.7,1.0,1.5,2.0,3.01,4.01,5.0,6.01,7.01,8.01,9.0,10.0]
snaps_ids = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,11,8,6,4]

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
})

basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (f"The voxel width is {dl} cMpc/h")

norm_dens = []
voids = []
filaments = []
nodes = []
walls = []
dens = []

with h5py.File('/Users/users/roana/roana/BSc_Thesis/nexus_outputs.h5', 'r') as f:
    for red in zs:
        norm_dens.append(np.transpose(f[f'z_{red}']['normalized_density'][:], (1,2,0)))
        dens.append(np.transpose(f[f'z_{red}']['density'][:], (1,2,0)))
        voids.append(np.transpose(f[f'z_{red}']['voids'][:], (1,2,0)))
        filaments.append(np.transpose(f[f'z_{red}']['filaments'][:], (1,2,0)))
        nodes.append(np.transpose(f[f'z_{red}']['nodes'][:], (1,2,0)))
        walls.append(np.transpose(f[f'z_{red}']['walls'][:], (1,2,0)))
        print(f"Suessfully Loaded Stuff for z={red}")

n = len(norm_dens)
print(f"{n} snapshots")

In [ ]:
ids_z0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['ParticleIDs'])
coords_z0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['Coordinates'])/1000

print("At redshift z=0...")

bool_filament = filaments[0].astype(bool)
bool_wall = walls[0].astype(bool)
bool_node = nodes[0].astype(bool)
bool_void = voids[0].astype(bool)

grid_x = np.floor(coords_z0[:, 0] / dl).astype(int) % res
grid_y = np.floor(coords_z0[:, 1] / dl).astype(int) % res
grid_z = np.floor(coords_z0[:, 2] / dl).astype(int) % res

in_node_mask = bool_node[grid_x, grid_y, grid_z]

node_particle_ids = ids_z0[in_node_mask]
print(f"Total DM particles: {len(ids_z0)}")
print(f"Particles in nodes: {len(node_particle_ids)}")

in_filament_mask = bool_filament[grid_x, grid_y, grid_z]

filament_particle_ids = ids_z0[in_filament_mask]
print(f"Particles in filaments: {len(filament_particle_ids)}")

in_wall_mask = bool_wall[grid_x, grid_y, grid_z]

wall_particle_ids = ids_z0[in_wall_mask]
print(f"Particles in walls: {len(wall_particle_ids)}")

in_void_mask = bool_void[grid_x, grid_y, grid_z]

void_particle_ids = ids_z0[in_void_mask]
print(f"Particles in voids: {len(void_particle_ids)}")

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

def update(frame):

    ax.clear()

    snap_id = frame

    particles_z = il.snapshot.loadSubset(basePath, snap_id, 'dm', ['Coordinates', 'ParticleIDs'])
    coords_z = particles_z['Coordinates']/1000
    ids_z = particles_z['ParticleIDs']

    match_mask = np.isin(ids_z, void_particle_ids)
    historical_coords = coords_z[match_mask]
    historical_coords = historical_coords[::100]

    z_min = 0.0 
    z_max = 35.0 

    # Extract Z coordinates and create a boolean mask
    z_coords = historical_coords[:, 2]
    slice_mask = (z_coords >= z_min) & (z_coords <= z_max)

    # Apply the mask
    x_slice = historical_coords[:, 0][slice_mask]
    y_slice = historical_coords[:, 1][slice_mask]

    ax.scatter(x_slice, y_slice, cmap = "inferno")
    ax.set_xlabel('cMpc/h')
    ax.set_ylabel('cMpc/h')
    ax.set_title('Forming Filaments')
    ax.set_aspect('equal');

anim = animation.FuncAnimation(fig, update, frames=99, interval=50)

anim.save('void_evol.mp4', writer='ffmpeg', fps=10)

print("2D Animation successfully saved!")

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)



snap_id = 99
particles_z = il.snapshot.loadSubset(basePath, snap_id, 'dm', ['Coordinates', 'ParticleIDs'])
coords_z = particles_z['Coordinates'][::1000]/1000
ids_z = particles_z['ParticleIDs'][::1000]
match_mask = np.isin(ids_z, void_particle_ids, invert=True)
historical_coords = coords_z[match_mask]
historical_coords = historical_coords
z_min = 0.0 
z_max = 35.0 
# Extract Z coordinates and create a boolean mask
z_coords = historical_coords[:, 2]
slice_mask = (z_coords >= z_min) & (z_coords <= z_max)
# Apply the mask
x_slice = historical_coords[:, 0][slice_mask]
y_slice = historical_coords[:, 1][slice_mask]
ax.scatter(x_slice, y_slice, s =0.1, alpha = 0.75, c="b", label = "Not Void")
match_mask = np.isin(ids_z, void_particle_ids)
historical_coords = coords_z[match_mask]
historical_coords = historical_coords
z_min = 0.0 
z_max = 35.0 
# Extract Z coordinates and create a boolean mask
z_coords = historical_coords[:, 2]
slice_mask = (z_coords >= z_min) & (z_coords <= z_max)
# Apply the mask
x_slice = historical_coords[:, 0][slice_mask]
y_slice = historical_coords[:, 1][slice_mask]
ax.scatter(x_slice, y_slice, s =0.1, alpha = 0.75, c="r", label = "Void")
ax.set_xlabel('cMpc/h')
ax.set_ylabel('cMpc/h')
ax.set_title('Forming Filaments')
ax.legend()
ax.set_aspect('equal');


In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter3D(x_slice, y_slice, z_coords, s =0.1, alpha = 0.75);


In [ ]:
ax.voxels(nodes[0].astype(bool))

plt.show()

In [ ]:
v_arr = np.array(voids)
print(np.shape(v_arr))

In [ ]:
n_arr = np.array(nodes)
print(np.shape(n_arr))

In [ ]:
plt.imshow(n_arr[0,:,:,76], origin = "lower");

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
slice = 37

def update(frame):

    ax.clear()

    ax.imshow(voids[17-frame][:,:,slice])
    ax.set_xlabel('cMpc/h')
    ax.set_ylabel('cMpc/h')
    ax.set_title(f'Void Evolution at slice {slice} at redshift {zs[17-frame]}')
    ax.set_aspect('equal');

anim = animation.FuncAnimation(fig, update, frames=18, interval=50)

anim.save('void_evol.mp4', writer='ffmpeg', fps=10)

print("2D Animation successfully saved!")

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

slice = 76

fig, ax = plt.subplots(figsize=(10, 8))

norm = mcolors.Normalize(vmin=1, vmax=4)
sm = cm.ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([]) 
cbar = fig.colorbar(sm, ax=ax, label='Cosmic Web Environment')

def update(frame):

    ax.clear()


    total = nodes[17-frame]*4 + filaments[17-frame]*3 + walls[17-frame]*2 + voids[17-frame]
    im = ax.imshow(total[:,:,slice], origin = "lower",vmax=4)
    ax.set_xlabel('cMpc/h')
    ax.set_ylabel('cMpc/h')
    ax.set_title(f'Structure Evolution at slice {slice} at redshift {zs[17-frame]}')
    ax.set_aspect('equal');

anim = animation.FuncAnimation(fig, update, frames=18, interval=50)

anim.save('struct_evol.mp4', writer='ffmpeg', fps=10)

print("2D Animation successfully saved!")